In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

members = pd.read_parquet(DATA_DIR / "members_v3.parquet")
train = pd.read_parquet(DATA_DIR / "train_v2.parquet")

print(f"Members shape: {members.shape}")
print(f"Train shape: {train.shape}")
print(f"\nMembers dtypes:\n{members.dtypes}")
print(f"\nMembers nulls:\n{members.isnull().sum()}")
print(f"\nSample:\n{members.head(3).to_string()}")

Members shape: (6769473, 6)
Train shape: (970960, 2)

Members dtypes:
msno                        str
city                      int64
bd                        int64
gender                      str
registered_via            int64
registration_init_time    int64
dtype: object

Members nulls:
msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64

Sample:
                                           msno  city  bd gender  registered_via  registration_init_time
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0    NaN              11                20110911
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0    NaN               7                20110914
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0    NaN              11                20110915


In [ ]:
REF_DATE = pd.Timestamp('2017-04-01')

members_train = members[members['msno'].isin(train['msno'])].copy()
print(f"Train users found in members: {len(members_train):,}")
print(f"Missing from members: {970960 - len(members_train):,}")

members_train['reg_date'] = pd.to_datetime(
    members_train['registration_init_time'].astype(str), format='%Y%m%d', errors='coerce'
)

# --- city ---
members_train['city'] = members_train['city'].astype('category')
members_train['is_city_1'] = (members_train['city'] == 1).astype(int)

# --- registered_via ---
members_train['registered_via'] = members_train['registered_via'].astype('category')

# --- age ---
members_train['age_known'] = (
    (members_train['bd'] >= 10) & (members_train['bd'] <= 80)
).astype(int)

members_train['bd_clean'] = members_train['bd'].where(
    (members_train['bd'] >= 10) & (members_train['bd'] <= 80), other=np.nan
)

# --- gender ---
members_train['gender_known'] = members_train['gender'].notna().astype(int)

# --- unknown_gender_young: use raw bd, not bd_clean ---
# EDA: unknown gender + bd < 25 (includes bd=0 unknowns and valid young ages) = 44.1% churn
members_train['unknown_gender_young'] = (
    (members_train['gender'].isna()) &
    (members_train['bd'] < 25)
).astype(int)

# --- account_age_days ---
members_train['account_age_days'] = (REF_DATE - members_train['reg_date']).dt.days

# --- is_invalid_registration ---
members_train['is_invalid_registration'] = (
    members_train['reg_date'] > pd.Timestamp('2017-03-31')
).astype(int)

# --- registration_year ---
members_train['registration_year'] = members_train['reg_date'].dt.year

print(f"\nInvalid registrations: {members_train['is_invalid_registration'].sum():,}")
print(f"Unknown gender young: {members_train['unknown_gender_young'].sum():,}")
print(f"Age known: {members_train['age_known'].sum():,}")
print(f"account_age_days nulls: {members_train['account_age_days'].isnull().sum():,}")
print(f"\nSample:\n{members_train[['msno','city','bd_clean','age_known','gender_known','unknown_gender_young','account_age_days','is_city_1','registered_via','is_invalid_registration','registration_year']].head(5).to_string()}")

Train users found in members: 860,967
Missing from members: 109,993

Invalid registrations: 1
Unknown gender young: 465,902
Age known: 386,393
account_age_days nulls: 0

Sample:
                                            msno city  bd_clean  age_known  gender_known  unknown_gender_young  account_age_days  is_city_1 registered_via  is_invalid_registration  registration_year
1   +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=    1       NaN          0             0                     1              2026          1              7                        0               2011
5   yLkV2gbZ4GLFwqTOXLVHz0VGrMYcgBGgKZ3kj9RiYu8=    4      30.0          1             1                     0              2024          0              9                        0               2011
9   I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=   13      63.0          1             1                     0              2022          0              9                        0               2011
10  OoDwiKZM+ZGr9P3fRivavg

In [ ]:
def age_bucket(age):
    if pd.isna(age):
        return 'unknown'
    elif age < 18:
        return 'under_18'
    elif age < 25:
        return '18_24'
    elif age < 35:
        return '25_34'
    elif age < 45:
        return '35_44'
    elif age < 55:
        return '45_54'
    else:
        return '55_plus'

members_train['bd_bucket'] = members_train['bd_clean'].apply(age_bucket).astype('category')

print(f"bd_bucket distribution:\n{members_train['bd_bucket'].value_counts()}")

feature_cols = [
    'msno',
    'city',
    'registered_via',
    'bd_clean',
    'age_known',
    'gender_known',
    'unknown_gender_young',
    'account_age_days',
    'is_city_1',
    'is_invalid_registration',
    'registration_year',
    'bd_bucket'
]

members_features = members_train[feature_cols].copy()

print(f"\nShape: {members_features.shape}")
print(f"\nNulls:\n{members_features.isnull().sum()}")

out_path = DATA_DIR / "members_features.parquet"
members_features.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")

bd_bucket distribution:
bd_bucket
unknown     474574
25_34       177130
18_24       104948
35_44        66310
45_54        22122
under_18      9693
55_plus       6190
Name: count, dtype: int64

Shape: (860967, 12)

Nulls:
msno                            0
city                            0
registered_via                  0
bd_clean                   474574
age_known                       0
gender_known                    0
unknown_gender_young            0
account_age_days                0
is_city_1                       0
is_invalid_registration         0
registration_year               0
bd_bucket                       0
dtype: int64
Saved: /Users/harshithnr/kkbox-churn/data/parquet/members_features.parquet
Size: 40.9 MB
Shape verify: (860967, 12)
